In [5]:
import numpy as np
import pandas as pd
import json
from hmmlearn import hmm
from sklearn.metrics import accuracy_score, classification_report

# -------------------------- 定义常量 --------------------------
hidden_states = ['CAPEC'+ str(i) for i in range(1,500)]
event_types = ['CAR'+ str(i) for i in range(1,1000)]

# 创建映射字典
event_to_code = {event: idx for idx, event in enumerate(event_types)}
state_to_code = {state: idx for idx, state in enumerate(hidden_states)}
code_to_state = {idx: state for state, idx in state_to_code.items()}
code_to_event = {idx: event for event, idx in event_to_code.items()}

# -------------------------- 数据加载和预处理 --------------------------
def load_dataset(filename):
    """加载数据集并转换为HMM可用的格式"""
    with open(filename, 'r') as f:
        sequences = json.load(f)
    
    # 转换数据格式
    X = []  # 观测序列
    lengths = []  # 序列长度
    y = []  # 状态序列
    
    for seq in sequences:
        # 转换事件和状态为数字编码
        events = [event_to_code[e] for e in seq['events']]
        states = [state_to_code[s] for s in seq['states']]
        
        X.extend(events)
        y.extend(states)
        lengths.append(len(events))
    
    return np.array(X).reshape(-1, 1), np.array(y), lengths

# -------------------------- HMM模型定义 --------------------------
class AttackStageHMM:
    def __init__(self):
        self.model = hmm.MultinomialHMM(
            n_components=499,  # 修改为499个状态
            algorithm="viterbi", 
            random_state=42,
            n_iter=100,
            verbose=True
        )
        
        # 初始化转移矩阵
        self.model.startprob_ = np.array([0.25, 0.25, 0.25, 0.25])
        
        # 从文件加载转移概率和观测概率矩阵
        with open('probs_matrix.json', 'r') as f:
            model_params = json.load(f)
            
        self.model.transmat_ = np.array(model_params['transition_probs'])
        self.model.emissionprob_ = np.array(model_params['emission_probs'])
    
    def fit(self, X, lengths):
        """训练模型"""
        self.model.fit(X, lengths=lengths)
        return self
    
    def predict(self, X, lengths):
        """预测状态序列"""
        return self.model.predict(X, lengths=lengths)
    
    def predict_proba(self, X, lengths):
        """预测状态概率"""
        return self.model.predict_proba(X, lengths=lengths)

# -------------------------- 评估函数 --------------------------
def evaluate_model(model, X, y, lengths):
    """评估模型性能"""
    # 预测状态序列
    y_pred = model.predict(X, lengths=lengths)
    
    # 计算整体准确率
    accuracy = accuracy_score(y, y_pred)
    
    # 获取实际出现的类别标签
    unique_labels = np.unique(np.concatenate([y, y_pred]))
    target_names = [hidden_states[i] for i in unique_labels]
    
    # 生成详细的分类报告
    report = classification_report(
        y, y_pred,
        labels=unique_labels,
        target_names=target_names,
        zero_division=0
    )
    
    return accuracy, report, y_pred

def predict_next_state(model, events):
    """预测下一个状态"""
    # 将事件转换为数字编码
    X = np.array([event_to_code[e] for e in events]).reshape(-1, 1)
    
    # 获取状态概率
    state_probs = model.predict_proba(X, lengths=[len(events)])[-1]
    
    # 获取最可能的状态
    predicted_state_idx = np.argmax(state_probs)
    
    return code_to_state[predicted_state_idx], state_probs

# -------------------------- 主程序 --------------------------
if __name__ == "__main__":
    # 加载数据
    print("加载数据...")
    X, y, lengths = load_dataset('attack_sequences.json')
    
    # 划分训练集和测试集
    total_sequences = len(lengths)
    train_size = int(0.8 * total_sequences)
    
    train_lengths = lengths[:train_size]
    test_lengths = lengths[train_size:]
    
    train_end_idx = sum(train_lengths)
    X_train = X[:train_end_idx]
    y_train = y[:train_end_idx]
    X_test = X[train_end_idx:]
    y_test = y[train_end_idx:]
    
    # 创建和训练模型
    print("训练HMM模型...")
    hmm_model = AttackStageHMM()
    hmm_model.fit(X_train, train_lengths)
    
    # 评估模型
    print("\n评估模型...")
    accuracy, report, predictions = evaluate_model(
        hmm_model.model, X_test, y_test, test_lengths
    )
    
    print(f"\n整体准确率: {accuracy*100:.2f}%")
    print("\n详细分类报告:")
    print(report)
    
    # 示例预测
    print("\n示例预测:")
    sample_sequence =  ['CAR'+ str(i) for i in range(1,5)]
    predicted_state, state_probs = predict_next_state(hmm_model.model, sample_sequence)
    
    print(f"输入序列: {sample_sequence}")
    print(f"预测的下一个攻击阶段: {predicted_state}")
    print("\n各状态概率:")
    for i, prob in enumerate(state_probs):
        print(f"{hidden_states[i]}: {prob:.3f}")
    
    # 保存模型参数
    model_params = {
        'transition_matrix': hmm_model.model.transmat_.tolist(),
        'emission_matrix': hmm_model.model.emissionprob_.tolist(),
        'initial_probabilities': hmm_model.model.startprob_.tolist(),
        'accuracy': accuracy
    }
    
    with open('hmm_model_params.json', 'w') as f:
        json.dump(model_params, f, indent=4)
    
    print("\n模型参数已保存到 'hmm_model_params.json'")

MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340
Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'transmat_' attribute is set, it will be overwritten during initialization because 'init_params' contains 't'
Fitting a model with 249000 free scalar parameters with only 8407 data points will result in a degenerate solution.


加载数据...
训练HMM模型...


         1       0.00000000             +nan
         2       0.00000000      -0.00000000



评估模型...

整体准确率: 0.05%

详细分类报告:
              precision    recall  f1-score   support

      CAPEC1       0.00      0.00      0.00       304
      CAPEC3       0.00      0.00      0.00         1
      CAPEC4       0.00      0.00      0.00         5
      CAPEC5       0.00      0.00      0.00        13
      CAPEC6       0.00      0.00      0.00         2
      CAPEC9       0.00      0.00      0.00         1
     CAPEC12       0.00      0.00      0.00         7
     CAPEC13       0.00      0.00      0.00         2
     CAPEC14       0.00      0.00      0.00         6
     CAPEC15       0.00      0.00      0.00         1
     CAPEC17       0.00      0.00      0.00        11
     CAPEC18       0.00      0.00      0.00         1
     CAPEC19       0.00      0.00      0.00         1
     CAPEC21       0.00      0.00      0.00         3
     CAPEC28       0.00      0.00      0.00         1
     CAPEC29       0.00      0.00      0.00        17
     CAPEC30       0.00      0.00      0.00      